<a href="https://colab.research.google.com/github/DeepthiManthapuram/Building_LLM_Applications/blob/main/Textbook_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain-community PyPDF

In [27]:
from langchain_community.document_loaders import WebBaseLoader
url = "https://www.deeplearningbook.org/contents/linear_algebra.html"
doc_loader = WebBaseLoader(url)
docs = doc_loader.load()
print(len(docs))


1


In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)
all_splits = text_splitter.split_documents(docs)
print(all_splits[0])

page_content='Chapter 2Linear AlgebraLinear algebra is a branch of mathematics that is widely used throughout scienceand engineering. Yet because linear algebra is a form of continuous rather thandiscrete mathematics, many computer scientists have little experience with it. Agood understanding of linear algebra is essential for understanding and workingwith many machine learning algorithms, especially deep learning algorithms. Wetherefore precede our introduction to deep learning with a focused presentation ofthe key linear algebra prerequisites.If you are already familiar with linear algebra, feel free to skip this chapter. Ifyou have previous experience with these concepts but need a detailed referencesheet to review key formulas, we recommend The Matrix Cookbook (Petersen andPedersen, 2006). If you have had no exposure at all to linear algebra, this chapterwill teach you enough to read this book, but we highly recommend that you alsoconsult another resource focused exclusively on te

In [29]:
print(all_splits[0].metadata)

{'source': 'https://www.deeplearningbook.org/contents/linear_algebra.html', 'title': '', 'language': 'No language found.'}


In [30]:
!pip install -qU langchain langchain-huggingface sentence-transformers

In [31]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
!pip install -qU chromadb langchain-chroma opentelemetry-api opentelemetry-sdk

In [ ]:
!pip install -qU opentelemetry-api opentelemetry-sdk

In [ ]:
!pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions

!pip install -U chromadb
!pip install -U opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions

Found existing installation: chromadb 1.5.9
Uninstalling chromadb-1.5.9:
  Successfully uninstalled chromadb-1.5.9
Found existing installation: opentelemetry-api 1.45.0
Uninstalling opentelemetry-api-1.45.0:
  Successfully uninstalled opentelemetry-api-1.45.0
Found existing installation: opentelemetry-sdk 1.45.0
Uninstalling opentelemetry-sdk-1.45.0:
  Successfully uninstalled opentelemetry-sdk-1.45.0
Found existing installation: opentelemetry-semantic-conventions 0.66b0
Uninstalling opentelemetry-semantic-conventions-0.66b0:
  Successfully uninstalled opentelemetry-semantic-conventions-0.66b0
  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached opentelemetry_api-1.45.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.45.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached opentelemetry_semantic_conventions-0.66b0-py3-none-any.whl.metadata (2.4 kB)
Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17

In [32]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name = "research_collection",
    embedding_function = embedding_model,
    persist_directory = "chroma_db"
)

document_ids = vector_store.add_documents(documents = all_splits)
print(len(document_ids))

54


In [33]:
def retrieve_context(query):
  retrieved_docs = vector_store.similarity_search(query, 2)

  docs_content = ""
  for doc in retrieved_docs:
    docs_content += f"source:{doc.metadata}\n"
    docs_content += f"content:{doc.page_content}\n"

  return docs_content, retrieved_docs

In [15]:
!pip install -U langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 16.5 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but y

In [34]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
model = init_chat_model(
    "google_genai:gemini-3.6-flash",
    api_key = api_key
)

In [38]:
def textbook_assistant(user_query, k=2):
  context, source_docs = retrieve_context(user_query)
  system_message = f"""You are a helpful chatbot.
                    Use only the following pieces of context to answer the
                    question. Don't makeup any new information: {context}"""
  messages = [
      {"role":"system", "content": system_message},
      {"role":"user", "content": user_query}
  ]

  response = model.invoke(messages)
  answer_text = response.content[0]["text"]

  return {
      "answer": answer_text,
      "source_documents": source_docs,
      "context_used": context
  }
result = textbook_assistant("what is linear algebra")

print(result["answer"])

Based on the provided context, linear algebra is a branch of continuous (rather than discrete) mathematics that is widely used throughout science and engineering. It is a fundamental mathematical discipline that is essential for understanding and working with many machine learning algorithms, especially deep learning algorithms.


In [41]:
print(result["context_used"])

source:{'source': 'https://www.deeplearningbook.org/contents/linear_algebra.html', 'title': '', 'language': 'No language found.'}
content:Chapter 2Linear AlgebraLinear algebra is a branch of mathematics that is widely used throughout scienceand engineering. Yet because linear algebra is a form of continuous rather thandiscrete mathematics, many computer scientists have little experience with it. Agood understanding of linear algebra is essential for understanding and workingwith many machine learning algorithms, especially deep learning algorithms. Wetherefore precede our introduction to deep learning with a focused presentation ofthe key linear algebra prerequisites.If you are already familiar with linear algebra, feel free to skip this chapter. Ifyou have previous experience with these concepts but need a detailed referencesheet to review key formulas, we recommend The Matrix Cookbook (Petersen andPedersen, 2006). If you have had no exposure at all to linear algebra, this chapterwill